# Step 0: event table, frozen splits, first reproducible baseline

Runs on Google Colab (CPU is enough for this step) or locally.

- The corpus (IEMP Malian Jembe, Polak, Tarsitani & Clayton 2018, OSF `M652X`) is **CC BY-NC-ND 4.0**. It is downloaded into the Colab session (`/content/corpus`), never into the repository.
- Outputs (event table, metrics, later model weights) go to your Google Drive under `MyDrive/djembe/`. Weights are not published while their licence status is unclear.

## 1. Code and pinned dependencies
If pip reports that numpy was replaced, use *Runtime > Restart session*, then continue from section 2.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
COMMIT = "ee54db53252ba9ced53dbaaa6a668598725d8b68"   # pinned code version; bump it when the code changes
if IN_COLAB and not os.path.exists(REPO):
    !git clone -q https://github.com/clementdumeril/Djembe.git "{REPO}" && git -C "{REPO}" checkout -q {COMMIT}
%pip install -q -r "{REPO}/requirements-train.txt"

## 2. Device and output folder

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
sys.path.insert(0, REPO); os.chdir(REPO)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    DEVICE = "cpu"
print("device:", DEVICE)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/djembe/step0"
    CORPUS = "/content/corpus"
else:
    OUT_DIR = os.path.join(REPO, "output", "step0")
    CORPUS = os.path.join(REPO, "corpus")
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["DJEMBE_CORPUS"] = CORPUS  # read by tests/test_splits.py
print("outputs ->", OUT_DIR)

## 3. Download the annotations from OSF (about 10 MB, audio not needed here)

In [ ]:
import djembe_dataset as d
d.download_corpus(CORPUS)

## 4. Build the event table
One row per annotated stroke, with its deviation from the isochronous grid. Built from the `_Onsets_Selected_Peak` files: the OSF copy of `MJ_Maraka_5_Onsets_Selected.csv` actually contains Woloso_5.

In [ ]:
events = d.build_events(CORPUS)
events_path = os.path.join(OUT_DIR, "events.csv")
events.to_csv(events_path, index=False)
print(len(events), "events,", events.take_id.nunique(), "takes")
events.groupby(["piece", "instrument"]).size().unstack(fill_value=0)

## 5. Leakage tests
The folds in `splits.json` are leave-one-take-out and versioned in the repository.

In [ ]:
!python -m pytest -q tests

## 6. Baseline: rhythm from the per-take D1 subdivision profile

In [ ]:
metrics_path = os.path.join(OUT_DIR, "step0_rf.json")
!python step0_rf_baseline.py --events "{events_path}" --out "{metrics_path}" 